# 04 — Colab L4: Qwen Scale-up + Gemma-2 Activation Capture

**런타임:** GPU **L4 (24GB VRAM) 필수**. Qwen2.5-3B / gemma-2-2b는 fp32에서 로컬(M1 8GB) 실행 불가 → L4 전용.

**목적:** 동결 매니페스트 `v1.0.3`(400 텍스트)에 대해 아키텍처 스코프 6모델 중 **미측정 4종**의 잔차 활성화를 캡처하여, 기존 `results/activations/`(gpt2 / pythia-160m / qwen2.5-0.5b, fp32)와 **동일 정밀도(fp32)** 로 통합한다.

**대상 모델 (fp32):**

| model_key | HF ID | 대략 파라미터 |
|---|---|---|
| qwen2.5-0.5b | Qwen/Qwen2.5-0.5B | 0.49B |
| qwen2.5-1.5b | Qwen/Qwen2.5-1.5B | 1.54B |
| qwen2.5-3b | Qwen/Qwen2.5-3B | 3.09B |
| gemma-2-2b | google/gemma-2-2b | 2.61B |

**등록 관측량 3종**
1. 길이보정 블록 업데이트 노름 `||h_(l+1) − h_l||_F / sqrt(T)`
2. 입력–업데이트 코사인 `cos(h_l, Δh_l)`
3. 출력/입력 노름비 `||h_(l+1)||_F / ||h_l||_F`

**불변 규칙 (AGENTS.md)**
- 동결 매니페스트 무결성 게이트 실패 시 **즉시 중단**.
- `human_review_status` 자동 완결 금지 / 동결 라벨 임의 부여 금지.
- 측정 산출물에 매니페스트 SHA-256 · 코드 리비전 · 모델 ID · 토크나이저 ID · 패키지 버전 · 실행 시각 기록.
- 서로 다른 `hidden_dim` 모델 간 **raw Frobenius 노름 비교 금지** → 무차원 비율 또는 RMS-per-element 사용.
- 깊이 비교는 항상 상대깊이 `rho_block = l/(L-1)` 사용.
- `gemma-2-2b`는 **게이트 모델** → HF 토큰(라이선스 동의) 필요.


In [ ]:
# ── 1. 런타임 환경 보고 (PyTorch 무단 업그레이드 금지 — AGENTS.md) ──
import sys, json, hashlib, platform
from datetime import datetime, timezone
import torch, transformers

print("python          :", sys.version.split()[0])
print("platform        :", platform.platform())
print("torch           :", torch.__version__)
print("transformers    :", transformers.__version__)
print("cuda available  :", torch.cuda.is_available())

GPU_NAME = None
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    GPU_NAME = p.name
    print("gpu             :", p.name)
    print("vram (GB)       :", round(p.total_memory / 1024**3, 1))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DTYPE = torch.float32  # 기존 gpt2/pythia/qwen0.5b 결과와 동일 정밀도 → 스케일 축 공정 비교
print("device          :", DEVICE, "| model dtype:", MODEL_DTYPE)
assert torch.cuda.is_available(), "L4 GPU 런타임이 아닙니다. 런타임 유형을 'L4 GPU'로 변경하세요."

## 1. 워크스페이스 연결 + 동결 매니페스트 무결성 게이트

동결 `v1.0.3`의 SHA-256 · 행수 · 텍스트 지문을 하드코딩 상수와 대조한다. 하나라도 불일치하면 캡처를 시작하지 않는다.


In [ ]:
# ── 2. 워크스페이스 연결 + 무결성 게이트 ──
from pathlib import Path
import io, csv, hashlib

# 동결 v1.0.3 기대 provenance 상수
EXPECTED_MANIFEST_SHA256 = "841aca615f3bd27a723e714fb2129af1f4b078155d1f169b591baaf88291d7e9"
EXPECTED_TEXT_FINGERPRINT = "3b3c049bc723e7e7c324c74975994271f7e7e9bb176b247ce621589ce56b43fc"
EXPECTED_ROWS = 400

PROJECT_ROOT = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    cand = Path("/content/drive/MyDrive/PhysLLM_N02_Antigravity_Workspace")
    if cand.exists():
        PROJECT_ROOT = cand
except Exception as e:
    print("drive mount skipped:", e)

if PROJECT_ROOT is None and Path("/content/PhysLLM_N02_Antigravity_Workspace").exists():
    PROJECT_ROOT = Path("/content/PhysLLM_N02_Antigravity_Workspace")
print("PROJECT_ROOT:", PROJECT_ROOT)

MANIFEST_PATH = None
if PROJECT_ROOT is not None:
    cand = PROJECT_ROOT / "data/frozen/N02_Main_Evaluation_FROZEN_Manifest_v1.0.3.csv"
    if cand.exists():
        MANIFEST_PATH = cand

if MANIFEST_PATH is None:
    from google.colab import files
    print("워크스페이스를 찾지 못했습니다. 매니페스트를 업로드하세요:")
    print("  N02_Main_Evaluation_FROZEN_Manifest_v1.0.3.csv")
    up = files.upload()
    MANIFEST_PATH = Path(next(iter(up)))

manifest_bytes = MANIFEST_PATH.read_bytes()
manifest_sha = hashlib.sha256(manifest_bytes).hexdigest()
rows = list(csv.DictReader(io.StringIO(manifest_bytes.decode("utf-8"))))
fingerprint = hashlib.sha256(
    "\n".join(f"{r['text_id']}\n{r['text']}" for r in rows).encode("utf-8")
).hexdigest()

print("manifest  :", MANIFEST_PATH.name)
print("  rows        :", len(rows))
print("  sha256      :", manifest_sha)
print("  fingerprint :", fingerprint)

assert len(rows) == EXPECTED_ROWS, f"행 수 불일치({len(rows)} != {EXPECTED_ROWS}) → 중단"
assert manifest_sha == EXPECTED_MANIFEST_SHA256, "매니페스트 SHA-256 불일치 → 중단"
assert fingerprint == EXPECTED_TEXT_FINGERPRINT, "텍스트 지문 불일치 → 중단"
print("\nOK — 무결성 게이트 통과 (동결 v1.0.3 확인)")

## 2. 잔차 스트림 캡처 코어 (등록 관측량 3종)

`gpt2`(transformer.h), `gpt_neox`(gpt_neox.layers), `llama/qwen/gemma`(model.layers)를 지원한다. 첫 블록 입력 `h_0`은 pre-hook, 각 블록 출력 `h_(l+1)`은 forward-hook으로 CPU에 fp64로 적재한다.


In [ ]:
# ── 3. 캡처 코어 ──
MODEL_REGISTRY = {
    "qwen2.5-0.5b": "Qwen/Qwen2.5-0.5B",
    "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B",
    "qwen2.5-3b":   "Qwen/Qwen2.5-3B",
    "gemma-2-2b":   "google/gemma-2-2b",
}


def resolve_transformer_blocks(model):
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    raise RuntimeError(f"지원하지 않는 아키텍처: {type(model).__name__}")


def first_hidden_tensor(output):
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, (tuple, list)):
        for item in output:
            if isinstance(item, torch.Tensor) and item.ndim == 3:
                return item
    raise TypeError(f"[B,T,D] hidden state 추출 실패: {type(output)}")


@torch.inference_mode()
def capture_single_text(tokenizer, model, model_key, manifest_row, device):
    enc = tokenizer(str(manifest_row["text"]), return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    n_tokens = int(enc["input_ids"].shape[1])

    blocks = resolve_transformer_blocks(model)
    captured = {}

    def pre_hook(module, args):
        captured[0] = args[0].detach().to("cpu", dtype=torch.float32).clone()[0]

    handles = [blocks[0].register_forward_pre_hook(pre_hook)]
    for idx, block in enumerate(blocks):
        def post_hook(module, args, output, i=idx):
            captured[i + 1] = (
                first_hidden_tensor(output).detach().to("cpu", dtype=torch.float32).clone()[0]
            )
        handles.append(block.register_forward_hook(post_hook))

    try:
        outputs = model(**enc, use_cache=False)
    finally:
        for h in handles:
            h.remove()

    n_blocks = len(blocks)
    if len(captured) != n_blocks + 1 or any(i not in captured for i in range(n_blocks + 1)):
        raise RuntimeError(f"상태 {n_blocks + 1}개 기대, {len(captured)}개 캡처")
    if not torch.isfinite(outputs.logits).all():
        raise FloatingPointError("비유한 logits")

    states = [captured[i].to(dtype=torch.float64) for i in range(n_blocks + 1)]
    for i, h in enumerate(states):
        if not torch.isfinite(h).all():
            raise FloatingPointError(f"레이어 {i} 비유한 hidden state")

    out_rows = []
    for l in range(n_blocks):
        h_in, h_out = states[l], states[l + 1]
        delta = h_out - h_in
        in_norm = torch.linalg.vector_norm(h_in).item()
        out_norm = torch.linalg.vector_norm(h_out).item()
        d_norm = torch.linalg.vector_norm(delta).item()
        dot = torch.sum(h_in * delta).item()
        cos = dot / (in_norm * d_norm) if in_norm > 0 and d_norm > 0 else float("nan")
        out_rows.append({
            "model_key": model_key,
            "text_id": str(manifest_row["text_id"]),
            "topic_id": str(manifest_row["topic_id"]),
            "topic_domain": str(manifest_row["topic_domain"]),
            "length_tier": str(manifest_row["length_tier"]),
            "style": str(manifest_row["style"]),
            "layer_idx": int(l),
            "n_layers": int(n_blocks),
            "rho_block": l / max(n_blocks - 1, 1),
            "n_tokens": n_tokens,
            "hidden_dim": int(h_in.shape[-1]),
            "input_norm_fro": in_norm,
            "output_norm_fro": out_norm,
            "update_norm_fro": d_norm,
            "update_norm_per_token": d_norm / max(n_tokens, 1) ** 0.5,
            "update_rms_per_element": d_norm / max(n_tokens * h_in.shape[-1], 1) ** 0.5,
            "input_update_cosine": cos,
            "output_input_norm_ratio": out_norm / max(in_norm, 1e-12),
        })

    meta = {"n_tokens": n_tokens, "n_layers": n_blocks, "hidden_dim": int(h_in.shape[-1])}
    return out_rows, meta

## 3. 캡처 실행 (증분 재개)

`results/activations/N02_activation_metrics.csv`에 **append**하고, `N02_capture_status.csv`를 원장으로 이미 완료된 `(model,text)` 쌍은 건너뛴다. 스크립트 `scripts/capture_activations.py`와 동일한 관측량 정의를 사용한다.

> `gemma-2-2b`는 게이트 모델이다. Colab 좌측 🔑(Secrets)에 `HF_TOKEN`을 추가하거나, 미리 라이선스에 동의해야 한다.


In [ ]:
# ── 4. 캡처 실행 ──
import pandas as pd
from datetime import datetime, timezone

try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        from huggingface_hub import login
        login(token=_tok)
        print("HF 로그인 완료 (userdata.HF_TOKEN)")
    else:
        print("경고: HF_TOKEN 미설정 → gemma-2-2b 접근 실패 가능")
except Exception as e:
    print("HF 로그인 생략:", e)

OUT_DIR = (PROJECT_ROOT / "results/activations") if PROJECT_ROOT is not None else Path("/content/results/activations")
OUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_CSV = OUT_DIR / "N02_activation_metrics.csv"
STATUS_CSV = OUT_DIR / "N02_capture_status.csv"
META_JSON = OUT_DIR / "N02_capture_run_metadata.json"

if METRICS_CSV.exists():
    _pre = pd.read_csv(METRICS_CSV)
    print("기존 metrics 존재:", len(_pre), "rows | models:", sorted(_pre['model_key'].unique()))
else:
    print("경고: 기존 metrics 없음 → 신규 생성 (gpt2/pythia baseline 별도 병합 필요)")


def _flush(batch_rows, status_rows):
    pd.DataFrame(batch_rows).to_csv(
        METRICS_CSV, mode="a" if METRICS_CSV.exists() else "w",
        header=not METRICS_CSV.exists(), index=False)
    pd.DataFrame(status_rows).to_csv(
        STATUS_CSV, mode="a" if STATUS_CSV.exists() else "w",
        header=not STATUS_CSV.exists(), index=False)


manifest_df = pd.DataFrame(rows)
completed = set()
if STATUS_CSV.exists():
    _st = pd.read_csv(STATUS_CSV)
    completed = set(zip(_st["model_key"].astype(str), _st["text_id"].astype(str)))
print("기존 완료 (model,text) 쌍:", len(completed))

RUN_STARTED = datetime.now(timezone.utc).isoformat()
per_model = {}

for model_key, hf_id in MODEL_REGISTRY.items():
    todo = [r for _, r in manifest_df.iterrows()
            if (model_key, str(r["text_id"])) not in completed]
    if not todo:
        print(f"[skip] {model_key}: 이미 완료")
        continue

    print(f"\n=== {model_key} ({hf_id}) — {len(todo)}/{len(manifest_df)} texts ===")
    tokenizer = transformers.AutoTokenizer.from_pretrained(hf_id)
    model = transformers.AutoModelForCausalLM.from_pretrained(
        hf_id, torch_dtype=MODEL_DTYPE, low_cpu_mem_usage=True
    ).to(DEVICE)
    model.eval()

    batch_rows, status_rows = [], []
    for i, row in enumerate(todo):
        mrows, meta = capture_single_text(tokenizer, model, model_key, row.to_dict(), DEVICE)
        batch_rows.extend(mrows)
        status_rows.append({
            "model_key": model_key,
            "text_id": str(row["text_id"]),
            "n_tokens": meta["n_tokens"],
            "status": "SUCCESS",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        })
        if len(status_rows) >= 10:
            _flush(batch_rows, status_rows)
            batch_rows, status_rows = [], []
            print(f"  진행 {i+1}/{len(todo)}", flush=True)
    if status_rows:
        _flush(batch_rows, status_rows)

    per_model[model_key] = {"hf_id": MODEL_REGISTRY[model_key]}
    print(f"[done] {model_key}")
    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

META_JSON.write_text(json.dumps({
    "manifest_sha256": manifest_sha,
    "text_fingerprint": fingerprint,
    "code": "notebooks/04_colab_l4_qwen_gemma_capture.ipynb",
    "device": str(DEVICE), "gpu_name": GPU_NAME, "model_dtype": str(MODEL_DTYPE),
    "python": sys.version.split()[0], "torch": torch.__version__,
    "transformers": transformers.__version__,
    "run_started_utc": RUN_STARTED,
    "run_finished_utc": datetime.now(timezone.utc).isoformat(),
    "models": MODEL_REGISTRY, "per_model": per_model,
}, indent=2, ensure_ascii=False))
print("\nrun metadata:", META_JSON)

## 4. 결과 검증 (행수 · 스키마 · 유한성)

모델별 행수 = `400 × n_layers`, NaN/Inf 부재, `SUCCESS` 상태를 확인한다.


In [ ]:
# ── 5. 검증 ──
import numpy as np
import pandas as pd

metrics = pd.read_csv(METRICS_CSV)
status = pd.read_csv(STATUS_CSV)
print("metrics rows:", len(metrics), "| status rows:", len(status), "\n")

summ = metrics.groupby("model_key").agg(
    hidden_dim=("hidden_dim", "max"),
    n_layers=("n_layers", "max"),
    n_texts=("text_id", "nunique"),
    rows=("layer_idx", "size"),
)
summ["expected_rows"] = summ["n_layers"] * summ["n_texts"]
print(summ.to_string(), "\n")

num_cols = ["input_norm_fro", "output_norm_fro", "update_norm_fro",
            "update_norm_per_token", "update_rms_per_element",
            "output_input_norm_ratio", "input_update_cosine"]
bad = ~np.isfinite(metrics[num_cols].to_numpy())
assert not bad.any(), f"비유한 값 {bad.sum()}개 발견 → 중단"
assert (status["status"] == "SUCCESS").all(), "실패 상태 행 존재"

for mk in MODEL_REGISTRY:
    g = metrics[metrics["model_key"] == mk]
    if len(g) == 0:
        print(f"[..] {mk}: 아직 미캡처")
        continue
    L = int(g["n_layers"].iloc[0]); n = int(g["text_id"].nunique())
    ok = len(g) == L * n
    print(f"{mk:14s} L={L:2d} texts={n:3d} rows={len(g):5d} {'OK' if ok else 'MISMATCH'}")
print("\n검증 통과")

## 5. 결과 내려받기

`results/activations/` 전체를 ZIP으로 묶어 내려받는다. Drive를 쓴 경우엔 이미 동기화되어 있으므로, 로컬 리포(`~/Downloads/physllmN02/...`)로 그대로 복사하면 된다.


In [ ]:
# ── 6. ZIP + 다운로드 ──
import shutil
zip_base = "/content/N02_results_qwen_gemma_capture"
shutil.make_archive(zip_base, "zip", root_dir=str(OUT_DIR))
print("zip:", zip_base + ".zip")
try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("download skipped:", e)

## 6. 이후 단계 (리포지토리 반영)

1. `results/activations/{N02_activation_metrics.csv, N02_capture_status.csv, N02_capture_run_metadata.json}` 를 로컬 리포 `~/Downloads/physllmN02/PhysLLM_N02_Antigravity_Workspace/results/activations/` 에 반영.
2. `scripts/cross_model_analysis.py` 로 6모델 통합 분석 재실행 (무차원 비율 · `rho_block` 기준).
3. `N02_Architecture_Scope_v1.1.md` 의 measured/not-measured 매트릭스 갱신.
4. 어떤 동결 매니페스트/해시가 확정 분석에 쓰이는지 provenance 확정 (R3 이슈).

**주의:** 이 노트북은 캡처·검증만 수행하며, 가설 판정(입증/기각) 문구를 생성하지 않는다.
